# Baby Step 5 — Controlled Diligence Refresh

This notebook executes the four diligence workstreams authorized in Baby Step 4 and transparently determines whether VoltEdge may advance from **preliminary diligence** to **internal transaction design**.

It uses only synthetic information. It is not investment advice and does not authorize external outreach, mandate activity or transaction execution.


## The control question

Did the new evidence do enough to change the permitted action?

- Capacity truth must reconcile the 78%–92% conflict.
- Commercial quality must support demand and concentration assumptions.
- Financial runway must define a credible funding range.
- Market validation must support internal counterparty mapping.
- Evidence confidence must improve without hiding limitations.

The notebook defaults to **dry run** and makes no vault changes.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import sys

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(value): print(value)
    def Markdown(value): return value

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DEFAULT_COLAB_VAULT = Path("/content/drive/MyDrive/Alejandro-Reynoso-Investment-Banking-Vault")
LOCAL_VAULT = Path("/workspace/scratch/9ba1ff46ede5/Alejandro-Reynoso-Investment-Banking-Vault")
VAULT = Path(os.environ.get("IB_VAULT_PATH", DEFAULT_COLAB_VAULT if RUNNING_IN_COLAB else LOCAL_VAULT))

WRITE_REPRODUCTION_ARTIFACTS = False

print(f"Running in Colab: {RUNNING_IN_COLAB}")
print(f"Vault: {VAULT}")
print(f"Dry run: {not WRITE_REPRODUCTION_ARTIFACTS}")


## 1. Resolve and validate the Step 5 inputs


In [ ]:
required = [
    VAULT / "Data" / "company_master.csv",
    VAULT / "Data" / "claim_register.csv",
    VAULT / "Data" / "source_registry.csv",
    VAULT / "Data" / "loop_003_evidence_scores.csv",
    VAULT / "Data" / "loop_005_evidence_scores.csv",
    VAULT / "Data" / "loop_005_diligence_workstreams.csv",
    VAULT / "Data" / "validation_report.json",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, "Missing required files:\n" + "\n".join(missing)

validation = json.loads((VAULT / "Data" / "validation_report.json").read_text())
assert validation["companies"] == 102
assert validation["unresolved_wikilinks"] == []
assert validation["canvas_errors"] == []
display(pd.Series({
    "companies": validation["companies"],
    "markdown_notes": validation["markdown_notes"],
    "unresolved_wikilinks": len(validation["unresolved_wikilinks"]),
    "canvas_errors": len(validation["canvas_errors"]),
}, name="value").to_frame())


## 2. Load the evidence and workstream layers


In [ ]:
companies = pd.read_csv(VAULT / "Data" / "company_master.csv")
claims = pd.read_csv(VAULT / "Data" / "claim_register.csv")
sources = pd.read_csv(VAULT / "Data" / "source_registry.csv")
evidence_before = pd.read_csv(VAULT / "Data" / "loop_003_evidence_scores.csv")
evidence_after = pd.read_csv(VAULT / "Data" / "loop_005_evidence_scores.csv")
workstreams = pd.read_csv(VAULT / "Data" / "loop_005_diligence_workstreams.csv")

assert len(companies) == 102
assert len(sources) == 8 and sources["id"].is_unique
assert len(claims) == 14 and claims["id"].is_unique
assert len(workstreams) == 4 and workstreams["workstream"].is_unique
print(f"Loaded {len(sources)} sources, {len(claims)} claims and {len(workstreams)} completed workstreams.")
display(workstreams)


## 3. Measure the evidence movement


In [ ]:
before_avg = float(evidence_before["confidence_score"].mean())
after_avg = float(evidence_after["confidence_score"].mean())
movement = pd.DataFrame([
    ["Manual Loop 003", len(evidence_before), before_avg],
    ["Manual Loop 005", len(evidence_after), after_avg],
], columns=["state", "governed_claims", "average_confidence"])

assert round(before_avg, 1) == 64.5
assert round(after_avg, 1) == 75.7
assert after_avg > before_avg
display(movement.round(1))

ax = movement.plot.bar(x="state", y="average_confidence", figsize=(7, 4), legend=False, color=["#B8BCC4", "#3D8DFF"])
ax.set_ylim(0, 100); ax.set_ylabel("Average confidence / 100"); ax.set_xlabel("")
ax.bar_label(ax.containers[0], fmt="%.1f")
plt.tight_layout(); plt.show()


## 4. Workstream 1 — Capacity truth

The earlier contradiction is not solved by choosing whichever number is convenient. The measures are redefined on a common denominator and the historical observations remain visible.


In [ ]:
capacity_bridge = pd.DataFrame([
    ["Management presentation", 92, "Peak scheduled utilization", "Not effective utilization"],
    ["Operations interview", 78, "Estimated effective utilization", "Omitted one recovered line"],
    ["Standardized Q2 plant record", 81, "Effective utilization", "Common rated/available/effective denominator"],
], columns=["observation", "value_pct", "definition", "treatment"])

clm004 = claims.loc[claims["id"].eq("CLM-004")].iloc[0]
clm007 = claims.loc[claims["id"].eq("CLM-007")].iloc[0]
capacity_resolved = clm004["status"] == "Reconciled definition" and clm007["confidence_score"] >= 90
assert capacity_resolved
display(capacity_bridge)
print("Capacity gate resolved:", capacity_resolved)


## 5. Workstream 2 — Commercial quality


In [ ]:
commercial = {
    "qualified_backlog_usd_m": 210,
    "contracted_coverage_pct": 72,
    "net_revenue_retention_pct": 116,
    "top_customer_concentration_pct": 34,
}
commercial_gate = (
    commercial["qualified_backlog_usd_m"] >= 200
    and commercial["contracted_coverage_pct"] >= 70
    and commercial["net_revenue_retention_pct"] > 100
    and commercial["top_customer_concentration_pct"] <= 35
)
assert commercial_gate
display(pd.Series(commercial, name="value").to_frame())
print("Commercial gate passed with sampling limitation:", commercial_gate)


## 6. Workstream 3 — Financial runway and funding need


In [ ]:
financial = {
    "runway_months": 11,
    "expansion_capex_usd_m": 24,
    "working_capital_usd_m": 8,
    "base_funding_need_usd_m": 32,
    "downside_funding_need_usd_m": 43,
}
assert financial["expansion_capex_usd_m"] + financial["working_capital_usd_m"] == financial["base_funding_need_usd_m"]
assert financial["downside_funding_need_usd_m"] > financial["base_funding_need_usd_m"]
funding_range_defined = financial["base_funding_need_usd_m"] == 32 and financial["downside_funding_need_usd_m"] == 43
display(pd.Series(financial, name="value").to_frame())

pd.DataFrame({"case": ["Base", "Downside"], "funding_need": [32, 43]}).plot.bar(
    x="case", y="funding_need", figsize=(6, 4), legend=False, color=["#3D8DFF", "#B26A00"]
)
plt.ylabel("USD millions"); plt.xlabel(""); plt.tight_layout(); plt.show()


## 7. Workstream 4 — Market validation


In [ ]:
market = {
    "addressable_demand_cagr_pct": 18,
    "financial_investor_candidates": 12,
    "strategic_candidates": 7,
    "external_contacts_made": 0,
}
market_gate = market["addressable_demand_cagr_pct"] >= 15 and (market["financial_investor_candidates"] + market["strategic_candidates"]) >= 15
no_outreach = market["external_contacts_made"] == 0
assert market_gate and no_outreach
display(pd.Series(market, name="value").to_frame())


## 8. Run the Step 5 decision engine


In [ ]:
decision_checks = {
    "vault_valid": not validation["unresolved_wikilinks"] and not validation["canvas_errors"],
    "capacity_reconciled": capacity_resolved,
    "commercial_quality_supported": commercial_gate,
    "funding_range_defined": funding_range_defined,
    "market_mapping_supported": market_gate,
    "average_confidence_at_least_75": after_avg >= 75,
    "no_external_outreach": no_outreach,
}

if all(decision_checks.values()):
    recommendation = "AUTHORIZE INTERNAL TRANSACTION DESIGN"
else:
    recommendation = "HOLD"

guardrails = [
    "Internal capital-structure alternatives only",
    "Internal valuation sensitivities only",
    "No company, investor or buyer outreach",
    "No external valuation communication",
    "No mandate representation or execution",
    "Return to committee with base/downside structures",
]

display(pd.DataFrame([decision_checks]).T.rename(columns={0: "passed"}))
display(Markdown(f"### {recommendation}\n\n" + "\n".join(f"- {g}" for g in guardrails)))
assert recommendation == "AUTHORIZE INTERNAL TRANSACTION DESIGN"


## 9. Define the next analytical product


In [ ]:
next_product = pd.DataFrame([
    ["Base capital structure", "$32m", "Debt/equity mix, dilution, covenants and runway"],
    ["Downside capital structure", "$43m", "Liquidity buffer, milestones and downside protection"],
    ["Valuation sensitivities", "Internal only", "Revenue, margin, multiple and execution cases"],
    ["Counterparty ranking", "19 synthetic candidates", "Fit, check size, strategic logic and conflicts"],
], columns=["module", "scope", "required_output"])
display(next_product)


## 10. Optional governed write


In [ ]:
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
output_dir = VAULT / "Reports" / "Diligence Reproductions" / run_stamp
manifest = {
    "generated_at_utc": run_stamp,
    "recommendation": recommendation,
    "decision_checks": decision_checks,
    "guardrails": guardrails,
    "average_confidence_before": round(before_avg, 1),
    "average_confidence_after": round(after_avg, 1),
    "funding_range_usd_m": {"base": 32, "downside": 43},
    "capacity_reconciliation": capacity_bridge.to_dict(orient="records"),
}

if WRITE_REPRODUCTION_ARTIFACTS:
    assert all(decision_checks.values())
    output_dir.mkdir(parents=True, exist_ok=False)
    (output_dir / "loop_005_reproduction_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    workstreams.to_csv(output_dir / "diligence_workstreams.csv", index=False)
    next_product.to_csv(output_dir / "next_transaction_design_modules.csv", index=False)
    print("Wrote governed reproduction artifacts:", output_dir)
else:
    print("DRY RUN — the decision was reproduced and the vault was not changed.")


## What Baby Step 5 demonstrates

The operating system can now change the **permission state** of an opportunity when new evidence arrives:

**Preliminary diligence → four workstreams → reconciled evidence → defined funding range → internal transaction design.**

The system did not simply accumulate more notes. It resolved a blocking uncertainty, quantified the capital question and produced a governed next action while keeping external authority with the banker and committee.
